In [ ]:
#Test 1: (g=t^2+x^3)
import torch

# Use float64 for this verification test
dtype = torch.float64

# Input columns: [t, x]
tx = torch.tensor(
    [
        [0.3,  0.4],
        [0.6, -0.5],
        [-0.2, 0.7]
    ],
    dtype=dtype,
    requires_grad=True
)

# Separate time and spatial coordinates
t = tx[:, 0:1]
x = tx[:, 1:2]

# Define the known mathematical function
g = t**2 + x**3

# First derivatives using automatic differentiation
first_derivatives = torch.autograd.grad(
    outputs=g,
    inputs=tx,
    grad_outputs=torch.ones_like(g),
    create_graph=True
)[0]

g_t = first_derivatives[:, 0:1]
g_x = first_derivatives[:, 1:2]

# Second derivative with respect to x
second_derivatives = torch.autograd.grad(
    outputs=g_x,
    inputs=tx,
    grad_outputs=torch.ones_like(g_x),
    create_graph=False
)[0]

g_xx = second_derivatives[:, 1:2]

# Analytical derivatives
expected_gt = 2*t
expected_gx = 3*x**2
expected_gxx = 6*x

# Display results
print("AUTOMATIC DIFFERENTIATION RESULTS")
print("---------------------------------")

print("Calculated g_t:", g_t.flatten().tolist())
print("Expected g_t:  ", expected_gt.flatten().tolist())

print("\nCalculated g_x:", g_x.flatten().tolist())
print("Expected g_x:  ", expected_gx.flatten().tolist())

print("\nCalculated g_xx:", g_xx.flatten().tolist())
print("Expected g_xx:  ", expected_gxx.flatten().tolist())

print("\nMAXIMUM ABSOLUTE ERRORS")

print("g_t error:",
      torch.max(torch.abs(g_t - expected_gt)).item())

print("g_x error:",
      torch.max(torch.abs(g_x - expected_gx)).item())

print("g_xx error:",
      torch.max(torch.abs(g_xx - expected_gxx)).item())

AUTOMATIC DIFFERENTIATION RESULTS
---------------------------------
Calculated g_t: [0.6, 1.2, -0.4]
Expected g_t:   [0.6, 1.2, -0.4]

Calculated g_x: [0.4800000000000001, 0.75, 1.4699999999999998]
Expected g_x:   [0.4800000000000001, 0.75, 1.4699999999999998]

Calculated g_xx: [2.4000000000000004, -3.0, 4.199999999999999]
Expected g_xx:   [2.4000000000000004, -3.0, 4.199999999999999]

MAXIMUM ABSOLUTE ERRORS
g_t error: 0.0
g_x error: 0.0
g_xx error: 0.0


In [ ]:
Test 2: (g=t^2+x^2)
import torch

dtype = torch.float64

# Input coordinates [t, x]
tx = torch.tensor(
    [
        [0.3,  0.4],
        [0.6, -0.5],
        [-0.2, 0.7]
    ],
    dtype=dtype,
    requires_grad=True
)

t = tx[:, 0:1]
x = tx[:, 1:2]

# Function
g = t**2 + x**2

# First derivatives
first_derivatives = torch.autograd.grad(
    outputs=g,
    inputs=tx,
    grad_outputs=torch.ones_like(g),
    create_graph=True
)[0]

g_t = first_derivatives[:, 0:1]
g_x = first_derivatives[:, 1:2]

# Second spatial derivative
second_derivatives = torch.autograd.grad(
    outputs=g_x,
    inputs=tx,
    grad_outputs=torch.ones_like(g_x),
    create_graph=False
)[0]

g_xx = second_derivatives[:, 1:2]

# Analytical derivatives
expected_gt = 2*t
expected_gx = 2*x
expected_gxx = torch.full_like(x, 2.0)

# Results
print("AUTOMATIC DIFFERENTIATION RESULTS")
print("---------------------------------")

print("Calculated g_t:", g_t.flatten().tolist())
print("Expected g_t:  ", expected_gt.flatten().tolist())

print("\nCalculated g_x:", g_x.flatten().tolist())
print("Expected g_x:  ", expected_gx.flatten().tolist())

print("\nCalculated g_xx:", g_xx.flatten().tolist())
print("Expected g_xx:  ", expected_gxx.flatten().tolist())

print("\nMAXIMUM ABSOLUTE ERRORS")

print("g_t error:",
      torch.max(torch.abs(g_t - expected_gt)).item())

print("g_x error:",
      torch.max(torch.abs(g_x - expected_gx)).item())

print("g_xx error:",
      torch.max(torch.abs(g_xx - expected_gxx)).item())

AUTOMATIC DIFFERENTIATION RESULTS
---------------------------------
Calculated g_t: [0.6, 1.2, -0.4]
Expected g_t:   [0.6, 1.2, -0.4]

Calculated g_x: [0.8, -1.0, 1.4]
Expected g_x:   [0.8, -1.0, 1.4]

Calculated g_xx: [2.0, 2.0, 2.0]
Expected g_xx:   [2.0, 2.0, 2.0]

MAXIMUM ABSOLUTE ERRORS
g_t error: 0.0
g_x error: 0.0
g_xx error: 0.0


In [ ]:
#Test 3: Three-point Burgers residual verification


#Test 4: Batched residual verification


#Test 5: Numerical precision comparison

"""B1.2A — Step 4: Burgers PDE residual verification

Purpose
-------
1. Verify the Burgers residual assembly on a known function g(t, x) = t^2 + x^3.
2. Compare Autograd derivatives/residuals with the analytical expressions.
3. Repeat the check in batches on the exact x and t coordinates from Burgers.npz.
4. Compare float32 and float64 residuals on the same coordinate grid.

This is a manufactured-function verification test. It does NOT train a PINN
and g(t, x) is NOT expected to solve the Burgers equation (so its residual is
not expected to be zero).
"""

from pathlib import Path
import math

import numpy as np
import torch


# ---------------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------------
NU = 0.01 / math.pi
TEST_DTYPE = torch.float64
BATCH_SIZE = 4096

# The training script originally stores the dataset here when ROOT=/content.
# If your file is elsewhere, change this path. If it does not exist, the cell
# will prompt you to upload the .npz file in Google Colab.
REFERENCE_PATH = Path("/content/reference_data/Burgers.npz")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Viscosity nu: {NU:.12f}")


# ---------------------------------------------------------------------
# 2. RESIDUAL VERIFICATION AT THREE TEST POINTS
# ---------------------------------------------------------------------
# Input order is [t, x], matching the Burgers PINN implementation.
tx = torch.tensor(
    [
        [0.3,  0.4],
        [0.6, -0.5],
        [-0.2, 0.7],
    ],
    dtype=TEST_DTYPE,
    device=DEVICE,
    requires_grad=True,
)

t = tx[:, 0:1]
x = tx[:, 1:2]
g = t**2 + x**3

# First derivatives: dg/dt and dg/dx.
first_derivatives = torch.autograd.grad(
    outputs=g,
    inputs=tx,
    grad_outputs=torch.ones_like(g),
    create_graph=True,
)[0]

g_t = first_derivatives[:, 0:1]
g_x = first_derivatives[:, 1:2]

# Second spatial derivative: d/dx(dg/dx).
# create_graph=False is sufficient because we do not differentiate this
# verification result again with respect to any trainable network weights.
second_derivatives = torch.autograd.grad(
    outputs=g_x,
    inputs=tx,
    grad_outputs=torch.ones_like(g_x),
    create_graph=False,
)[0]

g_xx = second_derivatives[:, 1:2]

# Burgers residual: R = g_t + g*g_x - nu*g_xx.
nu_tensor = tx.new_tensor(NU)
residual_autograd = g_t + g * g_x - nu_tensor * g_xx

# Analytical residual for g(t, x) = t^2 + x^3:
# g_t = 2t, g_x = 3x^2, g_xx = 6x.
residual_analytical = (
    2.0 * t
    + (t**2 + x**3) * (3.0 * x**2)
    - nu_tensor * (6.0 * x)
)

max_error_3points = torch.max(
    torch.abs(residual_autograd - residual_analytical)
).item()

print("\nTEST 3 — THREE-POINT BURGERS RESIDUAL CHECK")
print("-------------------------------------------")
print("Input tensor shape       :", tuple(tx.shape), "(Expected: (3, 2))")
print("g / derivative shapes    :", tuple(g.shape), tuple(g_t.shape), tuple(g_x.shape), tuple(g_xx.shape))
print("Residual tensor shape    :", tuple(residual_autograd.shape), "(Expected: (3, 1))")
print("Autograd residual        :", residual_autograd.detach().cpu().flatten().tolist())
print("Analytical residual      :", residual_analytical.detach().cpu().flatten().tolist())
print(f"Maximum absolute error   : {max_error_3points:.12e}")


# ---------------------------------------------------------------------
# 3. LOAD THE EXACT REFERENCE x AND t COORDINATES
# ---------------------------------------------------------------------
if not REFERENCE_PATH.exists():
    print(f"\nDataset not found at: {REFERENCE_PATH}")
    try:
        from google.colab import files

        print("Please upload the Burgers.npz file (the filename may also be Burgers(2).npz).")
        uploaded = files.upload()
        npz_names = [name for name in uploaded if name.lower().endswith(".npz")]
        if len(npz_names) != 1:
            raise RuntimeError(
                f"Expected exactly one uploaded .npz file, found {len(npz_names)}."
            )

        REFERENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
        uploaded_path = Path(npz_names[0])
        REFERENCE_PATH.write_bytes(uploaded_path.read_bytes())
    except ImportError as exc:
        raise FileNotFoundError(
            f"Could not find {REFERENCE_PATH}. Set REFERENCE_PATH to the location of Burgers.npz."
        ) from exc

with np.load(REFERENCE_PATH, allow_pickle=False) as data:
    required_arrays = {"x", "t", "usol"}
    missing = required_arrays.difference(data.files)
    if missing:
        raise ValueError(f"Dataset is missing required arrays: {sorted(missing)}")

    x_ref = np.asarray(data["x"], dtype=np.float64).reshape(-1)
    t_ref = np.asarray(data["t"], dtype=np.float64).reshape(-1)
    usol = np.asarray(data["usol"])

if usol.shape not in {(len(x_ref), len(t_ref)), (len(t_ref), len(x_ref))}:
    raise ValueError(
        f"Unexpected usol shape {usol.shape}; expected either "
        f"({len(x_ref)}, {len(t_ref)}) or ({len(t_ref)}, {len(x_ref)})."
    )

# Build the exact same coordinate combinations as the reference dataset.
# indexing='ij' gives T and X shape (number of times, number of x points).
T_grid, X_grid = np.meshgrid(t_ref, x_ref, indexing="ij")
tx_dense = np.column_stack((T_grid.reshape(-1), X_grid.reshape(-1)))

print("\nREFERENCE GRID")
print("--------------")
print("Dataset path             :", REFERENCE_PATH)
print("Number of x coordinates  :", len(x_ref))
print("Number of t coordinates  :", len(t_ref))
print("Number of (t, x) points  :", len(tx_dense))
print("Coordinate tensor order  : [t, x]")


# ---------------------------------------------------------------------
# 4. BATCHED RESIDUAL CHECK
# ---------------------------------------------------------------------
def calculate_batched_residuals(tx_values, dtype, batch_size=BATCH_SIZE):
    """Return Autograd residuals and max analytical mismatch for one dtype."""
    residual_batches = []
    max_analytical_error = 0.0

    for start in range(0, len(tx_values), batch_size):
        batch_values = tx_values[start : start + batch_size]
        batch = torch.as_tensor(
            batch_values,
            dtype=dtype,
            device=DEVICE,
        ).detach().clone().requires_grad_(True)

        tb = batch[:, 0:1]
        xb = batch[:, 1:2]
        gb = tb**2 + xb**3

        d1 = torch.autograd.grad(
            outputs=gb,
            inputs=batch,
            grad_outputs=torch.ones_like(gb),
            create_graph=True,
        )[0]
        gt = d1[:, 0:1]
        gx = d1[:, 1:2]

        d2 = torch.autograd.grad(
            outputs=gx,
            inputs=batch,
            grad_outputs=torch.ones_like(gx),
            create_graph=False,
        )[0]
        gxx = d2[:, 1:2]

        nu_local = batch.new_tensor(NU)
        r_autograd = gt + gb * gx - nu_local * gxx
        r_analytical = (
            2.0 * tb
            + (tb**2 + xb**3) * (3.0 * xb**2)
            - nu_local * (6.0 * xb)
        )

        batch_error = torch.max(torch.abs(r_autograd - r_analytical)).item()
        max_analytical_error = max(max_analytical_error, batch_error)

        # Convert stored residuals to float64 on CPU so the comparison between
        # float32 and float64 is carried out in a common representation.
        residual_batches.append(r_autograd.detach().cpu().to(torch.float64).numpy().reshape(-1))

    residual_all = np.concatenate(residual_batches)
    return residual_all, max_analytical_error


print("\nTEST 4 — BATCHED 25,600-POINT RESIDUAL CHECK")
print("---------------------------------------------")
print("Batch size               :", BATCH_SIZE)

residual_64, error_64 = calculate_batched_residuals(tx_dense, torch.float64)
residual_32, error_32 = calculate_batched_residuals(tx_dense, torch.float32)

max_precision_difference = float(np.max(np.abs(residual_32 - residual_64)))
mean_precision_difference = float(np.mean(np.abs(residual_32 - residual_64)))

print(f"Max |R_autograd - R_exact| (float64): {error_64:.12e}")
print(f"Max |R_autograd - R_exact| (float32): {error_32:.12e}")
print(f"Max |R_float32 - R_float64|         : {max_precision_difference:.12e}")
print(f"Mean |R_float32 - R_float64|        : {mean_precision_difference:.12e}")


# ---------------------------------------------------------------------
# 5. INTERPRETATION REMINDERS
# ---------------------------------------------------------------------
print("\nINTERPRETATION")
print("--------------")
print("1. The manufactured function g(t, x) = t^2 + x^3 is not expected to solve Burgers.")
print("   Therefore, its PDE residual should generally be NONZERO.")
print("2. This test verifies residual assembly/signs, viscosity usage, tensor shapes,")
print("   batching, and numerical agreement with the analytical residual.")
print("3. The float32/float64 comparison here is for this known function and this grid;")
print("   it does not measure the effect of precision on a trained PINN.")
print("4. A separate model-based test is still required after the trained checkpoint is available.")


Device: cpu
Viscosity nu: 0.003183098862

TEST 3 — THREE-POINT BURGERS RESIDUAL CHECK
-------------------------------------------
Input tensor shape       : (3, 2) (Expected: (3, 2))
g / derivative shapes    : (3, 1) (3, 1) (3, 1) (3, 1)
Residual tensor shape    : (3, 1) (Expected: (3, 1))
Autograd residual        : [0.666280562731589, 1.3857992965855137, 0.14964098478028057]
Analytical residual      : [0.666280562731589, 1.3857992965855137, 0.14964098478028057]
Maximum absolute error   : 0.000000000000e+00

Dataset not found at: /content/reference_data/Burgers.npz
Please upload the Burgers.npz file (the filename may also be Burgers(2).npz).


Saving Burgers(2).npz to Burgers(2).npz

REFERENCE GRID
--------------
Dataset path             : /content/reference_data/Burgers.npz
Number of x coordinates  : 256
Number of t coordinates  : 100
Number of (t, x) points  : 25600
Coordinate tensor order  : [t, x]

TEST 4 — BATCHED 25,600-POINT RESIDUAL CHECK
---------------------------------------------
Batch size               : 4096
Max |R_autograd - R_exact| (float64): 0.000000000000e+00
Max |R_autograd - R_exact| (float32): 0.000000000000e+00
Max |R_float32 - R_float64|         : 1.043586877358e-06
Mean |R_float32 - R_float64|        : 7.489564106934e-08

INTERPRETATION
--------------
1. The manufactured function g(t, x) = t^2 + x^3 is not expected to solve Burgers.
   Therefore, its PDE residual should generally be NONZERO.
2. This test verifies residual assembly/signs, viscosity usage, tensor shapes,
   batching, and numerical agreement with the analytical residual.
3. The float32/float64 comparison here is for this known function